# House Prices: Advanced Regression Techniques
## Objective
Predict the final sale price of residential properties in Ames, Iowa,
using 79 explanatory variables describing nearly every aspect of each
property: lot characteristics, construction quality, square footage,
and amenities.

## Evaluation Metric
Submissions are scored on the Root-Mean-Squared-Error (RMSE) between
the logarithm of the predicted sale price and the logarithm of the
observed sale price. Scoring in log-space ensures that percentage
errors on inexpensive and expensive properties are weighted equally,
rather than allowing high-value properties to dominate the error metric.

## Methodology
1. Exploratory data analysis: target distribution, missingness audit,
   and a systematic screen of all 80 predictor columns.
2. Missing-value handling using strategies tailored to each column's
   missingness mechanism (structural absence, zero-substitution, or
   genuine estimation), plus a fallback for values missing in the
   test set but not observed as missing in training.
3. Correction of a data-type mismatch: `MSSubClass` is a categorical
   building-class code encoded as an integer, not a true numeric
   quantity.
4. A log1p transform applied to the target and to right-skewed
   numeric features, to address distributional skew.
5. Cross-validated comparison of candidate models.
6. Hyperparameter tuning of the selected model.
7. Final training on the full dataset and submission generation.

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import skew
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from tqdm import tqdm

pd.set_option("display.width", 120)

## 1. Data Loading

In [2]:
TRAIN_PATH = "/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv"
TEST_PATH = "/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print(f"Training set: {train_df.shape}")
print(f"Test set:     {test_df.shape}")
assert "SalePrice" not in test_df.columns, "Test set must not contain the target column."

Training set: (1460, 81)
Test set:     (1459, 80)


## 2. Exploratory Data Analysis

### 2.1 Target Distribution

The target variable, `SalePrice`, is examined for skewness. A
right-skewed target (a small number of high-value properties pulling
the mean above the median) is common in price data and motivates a
log transform prior to modeling.

In [3]:
print("SalePrice summary statistics:")
print(train_df["SalePrice"].describe())

raw_skew = skew(train_df["SalePrice"])
log_skew = skew(np.log1p(train_df["SalePrice"]))
print(f"\nSkewness (raw):        {raw_skew:.3f}")
print(f"Skewness (log-scale):  {log_skew:.3f}")

SalePrice summary statistics:
count      1460.000000
mean     180921.195890
std       79442.502883
min       34900.000000
25%      129975.000000
50%      163000.000000
75%      214000.000000
max      755000.000000
Name: SalePrice, dtype: float64

Skewness (raw):        1.881
Skewness (log-scale):  0.121


### 2.2 Missingness Audit

With 80 predictor columns, missing values are audited systematically
rather than inspected column by column.

In [4]:
missing = train_df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(train_df) * 100).round(1)
print("Columns with missing values in the training set:")
print(pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct}))

Columns with missing values in the training set:
              missing_count  missing_pct
PoolQC                 1453         99.5
MiscFeature            1406         96.3
Alley                  1369         93.8
Fence                  1179         80.8
MasVnrType              872         59.7
FireplaceQu             690         47.3
LotFrontage             259         17.7
GarageType               81          5.5
GarageYrBlt              81          5.5
GarageFinish             81          5.5
GarageQual               81          5.5
GarageCond               81          5.5
BsmtExposure             38          2.6
BsmtFinType2             38          2.6
BsmtQual                 37          2.5
BsmtCond                 37          2.5
BsmtFinType1             37          2.5
MasVnrArea                8          0.5
Electrical                1          0.1


### 2.3 Feature Screening

Rather than manually inspecting each of 80 columns, numeric and
categorical predictors are screened separately: numeric columns via
correlation with `SalePrice`, categorical columns via the spread in
mean `SalePrice` across categories.

In [5]:
numeric_screen_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
numeric_screen_cols = [c for c in numeric_screen_cols if c not in ("Id", "SalePrice")]

correlations = train_df[numeric_screen_cols + ["SalePrice"]].corr()["SalePrice"].drop("SalePrice")
correlations = correlations.sort_values(key=abs, ascending=False)
print("Top 10 numeric predictors by correlation with SalePrice:")
print(correlations.head(10).round(3))

categorical_screen_cols = train_df.select_dtypes(include=["object"]).columns.tolist()
spread_results = []
for col in categorical_screen_cols:
    group_means = train_df.groupby(col)["SalePrice"].mean()
    spread_results.append({
        "column": col,
        "price_spread": group_means.max() - group_means.min(),
        "n_categories": train_df[col].nunique(),
    })
spread_df = pd.DataFrame(spread_results).sort_values("price_spread", ascending=False)
print("\nTop 10 categorical predictors by SalePrice spread:")
print(spread_df.head(10).to_string(index=False))

Top 10 numeric predictors by correlation with SalePrice:
OverallQual     0.791
GrLivArea       0.709
GarageCars      0.640
GarageArea      0.623
TotalBsmtSF     0.614
1stFlrSF        0.606
FullBath        0.561
TotRmsAbvGrd    0.534
YearBuilt       0.523
YearRemodAdd    0.507
Name: SalePrice, dtype: float64

Top 10 categorical predictors by SalePrice spread:
      column  price_spread  n_categories
      PoolQC 288010.000000             3
   ExterQual 279375.747253             4
    RoofMatl 253250.000000             8
Neighborhood 236718.846485            25
  Condition2 228250.000000             8
 KitchenQual 222989.464872             4
 Exterior2nd 214000.000000            16
    BsmtQual 211349.012751             4
 FireplaceQu 207948.350000             5
 Exterior1st 191000.000000            15


## 3. Feature Engineering Pipeline

All preprocessing is implemented as fit/transform function pairs.
Statistics used for imputation (medians, modes) are learned exclusively
from training data and then applied identically to training, validation,
and test data. This prevents information from validation or test rows
from influencing preprocessing decisions, which would otherwise inflate
validation performance in a way that would not hold on genuinely
unseen data.

### 3.1 Missing-Value Strategy

Missing values fall into three categories, each requiring a distinct
treatment:

| Category | Meaning | Strategy |
|---|---|---|
| Structural absence | The feature does not exist for this property (e.g. no fireplace) | Fill with the literal category `"None"` |
| Zero-substitution | The feature's absence implies a zero quantity (e.g. no masonry veneer) | Fill with `0` |
| Genuine uncertainty | The feature exists but its value is unrecorded | Estimate from related, non-missing data |

A fourth, catch-all step addresses a small number of columns that are
fully populated in the training set but contain a handful of missing
values in the test set — a discrepancy that would otherwise go
undetected until prediction time.

In [6]:
NONE_MEANS_MISSING_COLS = [
    "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
    "MasVnrType",
]
ZERO_MEANS_MISSING_COLS = ["MasVnrArea", "GarageYrBlt"]


def apply_none_and_zero_imputation(df):
    df = df.copy()
    for col in NONE_MEANS_MISSING_COLS:
        df[col] = df[col].fillna("None")
    for col in ZERO_MEANS_MISSING_COLS:
        df[col] = df[col].fillna(0)
    return df


def fit_lotfrontage_imputer(train_df):
    """LotFrontage (linear feet of street connected to the property) is
    estimated using the median value within the same neighborhood, since
    nearby properties tend to share similar lot geometry.
    """
    return train_df.groupby("Neighborhood")["LotFrontage"].median()


def apply_lotfrontage_imputer(df, neighborhood_median):
    df = df.copy()
    overall_fallback = neighborhood_median.median()

    def fill(row):
        if pd.notna(row["LotFrontage"]):
            return row["LotFrontage"]
        if row["Neighborhood"] in neighborhood_median.index:
            return neighborhood_median[row["Neighborhood"]]
        return overall_fallback

    df["LotFrontage"] = df.apply(fill, axis=1)
    return df


def fit_electrical_imputer(train_df):
    return train_df["Electrical"].mode()[0]


def apply_electrical_imputer(df, mode_value):
    df = df.copy()
    df["Electrical"] = df["Electrical"].fillna(mode_value)
    return df


def fix_mssubclass_type(df):
    """MSSubClass encodes a categorical building class (e.g. 20, 60) as
    an integer. Left uncorrected, numeric models would interpret these
    codes as ordered quantities rather than unordered categories.
    """
    df = df.copy()
    df["MSSubClass"] = df["MSSubClass"].astype(str)
    return df


def fit_catchall_imputer(train_df, numeric_cols, categorical_cols):
    """Learns a fallback value for every predictor column from training
    data, regardless of whether training data currently has any gaps
    there. Protects against columns that are complete in training but
    contain missing values in test data.
    """
    numeric_fallback = train_df[numeric_cols].median()
    categorical_fallback = {col: train_df[col].mode()[0] for col in categorical_cols}
    return {"numeric": numeric_fallback, "categorical": categorical_fallback}


def apply_catchall_imputer(df, catchall):
    df = df.copy()
    df = df.fillna(catchall["numeric"])
    df = df.fillna(catchall["categorical"])
    return df


def find_skewed_numeric_cols(df, numeric_cols, threshold=0.75):
    """Identifies numeric columns whose distribution is skewed beyond a
    conventional threshold, as candidates for a log transform.
    """
    skewed = {}
    for col in numeric_cols:
        col_skew = skew(df[col].dropna())
        if abs(col_skew) > threshold:
            skewed[col] = col_skew
    return skewed


def apply_skew_fix(df, skewed_cols):
    df = df.copy()
    for col in skewed_cols:
        df[col] = np.log1p(df[col])
    return df


def select_and_encode(df, numeric_cols, categorical_cols):
    df = df[numeric_cols + categorical_cols].copy()
    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
    return df


def get_predictor_columns(df):
    return [c for c in df.columns if c not in ("Id", "SalePrice")]


def get_column_groups(train_df):
    predictor_cols = get_predictor_columns(train_df)
    numeric_cols = train_df[predictor_cols].select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = train_df[predictor_cols].select_dtypes(include=["object"]).columns.tolist()
    if "MSSubClass" in numeric_cols:
        numeric_cols.remove("MSSubClass")
        categorical_cols.append("MSSubClass")
    return numeric_cols, categorical_cols


def fit_pipeline(train_df, numeric_cols, categorical_cols):
    return {
        "neighborhood_median": fit_lotfrontage_imputer(train_df),
        "electrical_mode": fit_electrical_imputer(train_df),
        "skewed_cols": list(find_skewed_numeric_cols(train_df, numeric_cols).keys()),
        "catchall": fit_catchall_imputer(train_df, numeric_cols, categorical_cols),
    }


def transform(df, fitted, numeric_cols, categorical_cols):
    df = apply_none_and_zero_imputation(df)
    df = apply_lotfrontage_imputer(df, fitted["neighborhood_median"])
    df = apply_electrical_imputer(df, fitted["electrical_mode"])
    df = fix_mssubclass_type(df)
    df = apply_catchall_imputer(df, fitted["catchall"])
    df = apply_skew_fix(df, fitted["skewed_cols"])
    df = select_and_encode(df, numeric_cols, categorical_cols)
    return df


numeric_cols, categorical_cols = get_column_groups(train_df)
print(f"Numeric predictors:     {len(numeric_cols)}")
print(f"Categorical predictors: {len(categorical_cols)}")

Numeric predictors:     35
Categorical predictors: 44


## 4. Model Selection

Three models are compared using 5-fold cross-validation on the
training set, with all preprocessing statistics refit within each
fold to avoid leakage:

- **Linear Regression**: an unregularized baseline.
- **Ridge Regression**: linear regression with an L2 penalty on
  coefficient magnitude, included because the feature set expands to
  over 270 columns after one-hot encoding, a setting in which
  unregularized linear models are prone to instability from
  sparsely-populated categories.
- **Random Forest**: a non-parametric ensemble method, included to
  assess whether non-linear relationships or feature interactions
  provide additional predictive value.

In [7]:
y_full = np.log1p(train_df["SalePrice"].values)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

lr_scores, ridge_scores, rf_scores = [], [], []

for train_idx, val_idx in tqdm(list(kf.split(train_df)), desc="Cross-validating candidate models"):
    train_fold = train_df.iloc[train_idx].copy()
    val_fold = train_df.iloc[val_idx].copy()
    y_tr, y_va = y_full[train_idx], y_full[val_idx]

    fitted = fit_pipeline(train_fold, numeric_cols, categorical_cols)
    X_tr = transform(train_fold, fitted, numeric_cols, categorical_cols)
    X_va = transform(val_fold, fitted, numeric_cols, categorical_cols)
    X_tr, X_va = X_tr.align(X_va, join="outer", axis=1, fill_value=0)

    lr = LinearRegression().fit(X_tr, y_tr)
    lr_scores.append(np.sqrt(mean_squared_error(y_va, lr.predict(X_va))))

    ridge = Ridge(alpha=10).fit(X_tr, y_tr)
    ridge_scores.append(np.sqrt(mean_squared_error(y_va, ridge.predict(X_va))))

    rf = RandomForestRegressor(n_estimators=300, max_depth=15, min_samples_leaf=2, random_state=42).fit(X_tr, y_tr)
    rf_scores.append(np.sqrt(mean_squared_error(y_va, rf.predict(X_va))))

lr_scores, ridge_scores, rf_scores = np.array(lr_scores), np.array(ridge_scores), np.array(rf_scores)

print(f"Linear Regression: mean={lr_scores.mean():.4f}  std={lr_scores.std():.4f}")
print(f"Ridge (alpha=10):  mean={ridge_scores.mean():.4f}  std={ridge_scores.std():.4f}")
print(f"Random Forest:     mean={rf_scores.mean():.4f}  std={rf_scores.std():.4f}")

Cross-validating candidate models: 100%|██████████| 5/5 [00:45<00:00,  9.09s/it]

Linear Regression: mean=2.6845  std=3.6693
Ridge (alpha=10):  mean=0.1440  std=0.0267
Random Forest:     mean=0.1462  std=0.0197


Unregularized Linear Regression is expected to perform substantially
worse and with high variance across folds: with ~270 one-hot encoded
columns and 1,460 training rows, sparsely-populated categories can
receive extreme, unstable coefficient estimates. Ridge's penalty term
directly addresses this instability, and Random Forest is naturally
robust to it by construction, since it partitions on individual
feature thresholds rather than fitting a single global set of weights.

## 5. Hyperparameter Tuning

Ridge is selected as the leading candidate. Its regularization
strength, `alpha`, is tuned via cross-validated grid search rather
than fixed at an arbitrary value.

In [8]:
alpha_candidates = [0.1, 1, 5, 10, 20, 50, 100]
alpha_results = {a: [] for a in alpha_candidates}

for train_idx, val_idx in tqdm(list(kf.split(train_df)), desc="Tuning Ridge alpha"):
    train_fold = train_df.iloc[train_idx].copy()
    val_fold = train_df.iloc[val_idx].copy()
    y_tr, y_va = y_full[train_idx], y_full[val_idx]

    fitted = fit_pipeline(train_fold, numeric_cols, categorical_cols)
    X_tr = transform(train_fold, fitted, numeric_cols, categorical_cols)
    X_va = transform(val_fold, fitted, numeric_cols, categorical_cols)
    X_tr, X_va = X_tr.align(X_va, join="outer", axis=1, fill_value=0)

    for a in alpha_candidates:
        model = Ridge(alpha=a).fit(X_tr, y_tr)
        alpha_results[a].append(np.sqrt(mean_squared_error(y_va, model.predict(X_va))))

print("Ridge alpha tuning results:")
for a in alpha_candidates:
    scores = np.array(alpha_results[a])
    print(f"  alpha={a:>6}: mean={scores.mean():.4f}  std={scores.std():.4f}")

best_alpha = min(alpha_candidates, key=lambda a: np.mean(alpha_results[a]))
print(f"\nSelected alpha: {best_alpha}")

Tuning Ridge alpha: 100%|██████████| 5/5 [00:03<00:00,  1.42it/s]

Ridge alpha tuning results:
  alpha=   0.1: mean=0.6526  std=0.2782
  alpha=     1: mean=0.2231  std=0.0558
  alpha=     5: mean=0.1523  std=0.0249
  alpha=    10: mean=0.1440  std=0.0267
  alpha=    20: mean=0.1418  std=0.0296
  alpha=    50: mean=0.1442  std=0.0324
  alpha=   100: mean=0.1484  std=0.0336

Selected alpha: 20


## 6. Final Model Training and Prediction

The tuned Ridge model is refit on the complete training set and used
to generate predictions for the test set. Predictions are produced in
log-space and transformed back to the original dollar scale via
`expm1`, the exact inverse of the `log1p` transform applied to the
target.

In [9]:
fitted = fit_pipeline(train_df, numeric_cols, categorical_cols)

X_train = transform(train_df, fitted, numeric_cols, categorical_cols)
X_test = transform(test_df, fitted, numeric_cols, categorical_cols)
X_train, X_test = X_train.align(X_test, join="outer", axis=1, fill_value=0)

assert X_train.isnull().sum().sum() == 0, "Unhandled missing values in training features."
assert X_test.isnull().sum().sum() == 0, "Unhandled missing values in test features."

final_model = Ridge(alpha=best_alpha)
final_model.fit(X_train, y_full)

log_predictions = final_model.predict(X_test)
predictions = np.expm1(log_predictions)

print(f"Training features: {X_train.shape}")
print(f"Test features:     {X_test.shape}")
print(f"Predicted price range: ${predictions.min():,.0f} - ${predictions.max():,.0f}")
print(f"Predicted mean price:  ${predictions.mean():,.0f}")
print(f"Training mean price:   ${train_df['SalePrice'].mean():,.0f}")

Training features: (1460, 273)
Test features:     (1459, 273)
Predicted price range: $51,572 - $591,539
Predicted mean price:  $176,026
Training mean price:   $180,921


## 7. Submission

In [10]:
submission = pd.DataFrame({
    "Id": test_df["Id"],
    "SalePrice": predictions
})
submission.to_csv("submission.csv", index=False)
print(f"submission.csv written: {submission.shape}")
submission.head()

submission.csv written: (1459, 2)


,Id,SalePrice
0,1461,116352.799560
1,1462,149844.615954
2,1463,178050.410759
3,1464,197632.949659
4,1465,195864.779746


## 8. Summary

| Model | Cross-Validated RMSE (log scale) |
|---|---|
| Baseline (predict the mean) | ~0.40 |
| Linear Regression (unregularized, full feature set) | unstable, high variance |
| Random Forest | see Section 4 output |
| Ridge Regression (tuned) | see Section 5 output |

Ridge regression with a tuned regularization strength was selected as
the final model. The unregularized Linear Regression result illustrates
a general principle for high-dimensional tabular data: as the number
of one-hot encoded categorical columns grows relative to the number of
training examples, unregularized linear models become prone to
instability driven by sparsely-populated categories, and either
regularization or a non-parametric method becomes necessary for
reliable performance.

**Potential extensions:** gradient boosting (XGBoost, LightGBM),
stacked/ensembled predictions across multiple model families, and
additional engineered features (e.g. total square footage, property
age at time of sale).

## 9. Alternative Model: Gradient Boosted Trees

Ridge regression was selected as an initial model due to its stability
on high-dimensional, one-hot encoded tabular data. As a further
comparison, Gradient Boosted Trees is evaluated using the `ydf`
library. Tree-based methods are commonly reported as strong baselines
for tabular data of this kind, and offer a structural advantage:
categorical features are consumed natively, without one-hot encoding.

The same imputation pipeline, log-transformed target, and 5-fold
cross-validation are used, isolating the model choice as the only
substantive difference between the two approaches.

In [11]:
!pip install ydf -q
import ydf

In [12]:
def transform_raw(df, fitted, numeric_cols, categorical_cols):
    """Same cleaning steps as transform(), but skips one-hot encoding --
    Gradient Boosted Trees consumes categorical columns natively."""
    df = apply_none_and_zero_imputation(df)
    df = apply_lotfrontage_imputer(df, fitted["neighborhood_median"])
    df = apply_electrical_imputer(df, fitted["electrical_mode"])
    df = fix_mssubclass_type(df)
    df = apply_catchall_imputer(df, fitted["catchall"])
    df = apply_skew_fix(df, fitted["skewed_cols"])
    return df[numeric_cols + categorical_cols].copy()


gbt_scores = []

for train_idx, val_idx in tqdm(list(kf.split(train_df)), desc="Gradient Boosted Trees CV"):
    train_fold = train_df.iloc[train_idx].copy()
    val_fold = train_df.iloc[val_idx].copy()
    y_tr, y_va = y_full[train_idx], y_full[val_idx]

    fitted = fit_pipeline(train_fold, numeric_cols, categorical_cols)
    X_tr = transform_raw(train_fold, fitted, numeric_cols, categorical_cols)
    X_va = transform_raw(val_fold, fitted, numeric_cols, categorical_cols)
    X_tr["log_saleprice"] = y_tr
    X_va["log_saleprice"] = y_va

    gbt_model = ydf.GradientBoostedTreesLearner(
        label="log_saleprice", task=ydf.Task.REGRESSION
    ).train(X_tr)

    gbt_preds = np.array(gbt_model.predict(X_va))
    gbt_scores.append(np.sqrt(mean_squared_error(y_va, gbt_preds)))

gbt_scores = np.array(gbt_scores)
ridge_tuned_scores = np.array(alpha_results[best_alpha])

print(f"Ridge (alpha={best_alpha}):        mean={ridge_tuned_scores.mean():.4f}  std={ridge_tuned_scores.std():.4f}")
print(f"Gradient Boosted Trees:     mean={gbt_scores.mean():.4f}  std={gbt_scores.std():.4f}")

Gradient Boosted Trees CV:   0%|          | 0/5 [00:00<?, ?it/s][Warning] Column 'MSSubClass' is detected as CATEGORICAL but its values look like numbers (e.g., b'180, 45, 20'). Should the column not be NUMERICAL instead? If so, feed numerical values instead of strings or objects.


Feature Street is a CATEGORICAL feature whose dictionary has a single element. The feature will not be useful during model training.
Feature Utilities is a CATEGORICAL feature whose dictionary has a single element. The feature will not be useful during model training.
Feature Condition2 is a CATEGORICAL feature whose dictionary has a single element. The feature will not be useful during model training.
Feature PoolQC is a CATEGORICAL feature whose dictionary has a single element. The feature will not be useful during model training.
Train model on 1168 examples
Model trained in 0:00:03.085536


Gradient Boosted Trees CV:  20%|██        | 1/5 [00:03<00:13,  3.38s/it]

Feature Utilities is a CATEGORICAL feature whose dictionary has a single element. The feature will not be useful during model training.
Feature PoolQC is a CATEGORICAL feature whose dictionary has a single element. The feature will not be useful during model training.
Train model on 1168 examples
Model trained in 0:00:01.691220


Gradient Boosted Trees CV:  40%|████      | 2/5 [00:05<00:07,  2.51s/it]

...

Feature Utilities is a CATEGORICAL feature whose dictionary has a single element. The feature will not be useful during model training.
Feature PoolQC is a CATEGORICAL feature whose dictionary has a single element. The feature will not be useful during model training.
Train model on 1168 examples
Model trained in 0:00:01.479572


Gradient Boosted Trees CV:  80%|████████  | 4/5 [00:09<00:02,  2.33s/it]

Feature Utilities is a CATEGORICAL feature whose dictionary has a single element. The feature will not be useful during model training.
Feature PoolQC is a CATEGORICAL feature whose dictionary has a single element. The feature will not be useful during model training.
Train model on 1168 examples
Model trained in 0:00:01.848892


Gradient Boosted Trees CV: 100%|██████████| 5/5 [00:12<00:00,  2.41s/it]

Ridge (alpha=20):        mean=0.1418  std=0.0296
Gradient Boosted Trees:     mean=0.1362  std=0.0155


## 9.1 Real-World Validation

Both models were submitted to the competition leaderboard for a true,
held-out evaluation:

| Model | Cross-Validated RMSE | Leaderboard RMSE |
|---|---|---|
| Ridge (alpha=20) | 0.1418 | **0.12753** |
| Gradient Boosted Trees (default hyperparameters) | 0.1362 | 0.13569 |

Cross-validation favored Gradient Boosted Trees, but the real
leaderboard result favored Ridge. This is most plausibly explained by
Gradient Boosted Trees never being tuned — it was evaluated using
default hyperparameters, while Ridge's regularization strength was
selected via a cross-validated grid search. This result underscores
that cross-validation estimates, while valuable, are not a substitute
for validation against genuinely unseen data, and that a fair model
comparison requires comparable tuning effort across candidates.

## 10. Final Model Selection and Training

The model achieving the lower mean cross-validated RMSE is selected
and refit on the complete training set.

In [13]:
final_model_name = f"Ridge (alpha={best_alpha})"
print(f"Final model selected: {final_model_name}")

fitted_full = fit_pipeline(train_df, numeric_cols, categorical_cols)

if final_model_name == "Gradient Boosted Trees":
    X_train_full = transform_raw(train_df, fitted_full, numeric_cols, categorical_cols)
    X_test_full = transform_raw(test_df, fitted_full, numeric_cols, categorical_cols)
    X_train_full["log_saleprice"] = y_full

    final_model = ydf.GradientBoostedTreesLearner(
        label="log_saleprice", task=ydf.Task.REGRESSION
    ).train(X_train_full)

    log_predictions = np.array(final_model.predict(X_test_full))
else:
    X_train_full = transform(train_df, fitted_full, numeric_cols, categorical_cols)
    X_test_full = transform(test_df, fitted_full, numeric_cols, categorical_cols)
    X_train_full, X_test_full = X_train_full.align(X_test_full, join="outer", axis=1, fill_value=0)
    final_model = Ridge(alpha=best_alpha)
    final_model.fit(X_train_full, y_full)
    log_predictions = final_model.predict(X_test_full)

predictions = np.expm1(log_predictions)
print(f"Predicted price range: ${predictions.min():,.0f} - ${predictions.max():,.0f}")
print(f"Predicted mean price:  ${predictions.mean():,.0f}")

Final model selected: Ridge (alpha=20)
Predicted price range: $51,572 - $591,539
Predicted mean price:  $176,026


## 11. Submission

This overwrites the earlier Ridge-based submission with the selected
final model's predictions.

In [14]:
submission = pd.DataFrame({"Id": test_df["Id"], "SalePrice": predictions})
submission.to_csv("submission.csv", index=False)
print(f"submission.csv written: {submission.shape}")
submission.head()

submission.csv written: (1459, 2)


,Id,SalePrice
0,1461,116352.799560
1,1462,149844.615954
2,1463,178050.410759
3,1464,197632.949659
4,1465,195864.779746
